# 02 Stationarity and the evaluation structure

This notebook answers two questions with tests rather than assertions.

1. Why does this project model returns instead of prices? The answer is a
   stationarity argument, and it is checked here with ADF and KPSS on both
   series for every asset.
2. What linear structure is left in the return series once the trend is
   removed, and where does the remaining structure actually sit? This is checked
   with ACF, PACF and Ljung-Box on returns and on squared returns.

It closes with the walk forward fold structure that every result in this project
is measured on, drawn from the real feature matrix index rather than from a
description of it.

**Run this notebook from the repository root**, so that `from src...` imports
resolve. If it is launched from inside `notebooks/`, the first code cell walks
one directory up to find the repository root.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import schema
from src.config import load_config
from src.data.build_panel import load_auxiliary, load_panel
from src.features.registry import build_features
from src.models.arima import stationarity_tests
from src.splits.walk_forward import splitter_from_config

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

panel = load_panel(ROOT / "data" / "interim" / "panel_1d.parquet")
config = load_config(ROOT / "config" / "base.yaml")

assets = list(panel.index.get_level_values(schema.ASSET).unique())
close_wide = panel[schema.CLOSE].unstack(level=schema.ASSET).sort_index()[list(assets)]
log_price = np.log(close_wide)
log_return = log_price.diff()

print("repository root:", ROOT)
print("assets:", assets)

## Stationarity of log prices and log returns

A stationary series has a mean, variance and autocovariance structure that do not
depend on where in time you look. Almost every estimator used later assumes this,
including ordinary least squares standard errors, ARIMA, and the implicit
assumption that a model fitted on 2018 to 2022 tells you something about 2023.

A log price series is a random walk with drift, so its variance grows with time
and its mean is wherever the walk happens to have wandered. Fitting a model to it
produces a high R squared that comes entirely from the level, not from any
forecast. The first difference of a random walk is stationary, which is why the
targets in `src.data.targets` are all built from differences.

The table runs `src.models.arima.stationarity_tests` on both series for every
asset. That function is the same one the ARIMA model uses, so these numbers are
the ones the modelling code sees.

In [ ]:
rows = []
for asset in assets:
    price_series = log_price[asset].dropna()
    return_series = log_return[asset].dropna()
    for label, series in (("log price", price_series), ("log return", return_series)):
        result = stationarity_tests(series)
        rows.append({
            "asset": asset,
            "series": label,
            "n": len(series),
            "adf_stat": result["adf_stat"],
            "adf_pvalue": result["adf_pvalue"],
            "kpss_stat": result["kpss_stat"],
            "kpss_pvalue": result["kpss_pvalue"],
        })

stationarity = pd.DataFrame(rows)
stationarity["adf_rejects_unit_root"] = stationarity["adf_pvalue"] < 0.05
stationarity["kpss_rejects_stationarity"] = stationarity["kpss_pvalue"] < 0.05
stationarity["verdict"] = np.where(
    stationarity["adf_rejects_unit_root"] & ~stationarity["kpss_rejects_stationarity"],
    "stationary, both tests agree",
    np.where(
        ~stationarity["adf_rejects_unit_root"] & stationarity["kpss_rejects_stationarity"],
        "non stationary, both tests agree",
        "tests disagree, inconclusive",
    ),
)
stationarity.set_index(["asset", "series"]).round(4)

### How to read the two tests together

The two tests are set up in opposite directions, and that is the point of running
both.

- ADF has a null hypothesis of a unit root, that is, of non stationarity. A small
  p-value rejects the unit root and therefore argues for stationarity.
- KPSS has a null hypothesis of stationarity around a constant. A small p-value
  rejects stationarity and therefore argues for a unit root.

Neither test on its own is decisive, because failing to reject a null is not
evidence for it. A test can fail to reject simply because it has no power on the
sample available. The informative case is agreement:

- ADF rejects and KPSS does not reject: both point to stationarity. This is the
  expected outcome for the return series.
- ADF does not reject and KPSS rejects: both point to a unit root. This is the
  expected outcome for the log price series.
- Any other combination means the sample does not settle the question, and the
  right response is to say so rather than to pick the answer that suits.

Three practical caveats apply to the numbers above.

The KPSS p-value reported by statsmodels is clipped to the table range, so values
printed as 0.01 or 0.10 mean "at or beyond this bound" rather than an exact
figure.

A rejection by ADF is a statement about a unit root specifically, not a
certificate that the series is well behaved in every other respect. The return
series is stationary in mean but its variance is strongly time varying, which is
exactly what the next section is about.

An individual price series can still produce a marginal ADF rejection. A log
price that happens to end the sample near where it started, having traded in a
wide range in between, gives the test something that looks like mean reversion,
and a p-value just under 0.05 on one asset out of four is the kind of result a
5 percent test produces by construction. Check the verdict column rather than
assuming the expected pattern holds everywhere, and treat a single marginal
rejection as weak evidence rather than as grounds for modelling that asset in
levels while modelling the others in returns.

## Autocorrelation of returns and of squared returns

ACF measures the total correlation between a series and its own lag. PACF
measures the part of that correlation not already explained by the shorter lags,
which is what identifies an autoregressive order.

Both are computed on returns and on squared returns. Squaring removes the sign
and leaves the magnitude, so the squared series measures activity rather than
direction. The comparison between the two rows for each asset is the empirical
basis for the whole modelling split in this project: if the return ACF stays near
zero and the squared return ACF does not, then the sign is close to
unforecastable while the magnitude is not.

The shaded band in each panel is the approximate 95 percent interval under the
null of no autocorrelation. With several thousand observations that band is
narrow, so a bar only slightly outside it is not a usable signal.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

MAX_LAG = 30

figure, axes = plt.subplots(len(assets), 4, figsize=(17, 3.2 * len(assets)))
for row, asset in enumerate(assets):
    series = log_return[asset].dropna()
    squared = series.pow(2)

    plot_acf(series, lags=MAX_LAG, ax=axes[row, 0], zero=False)
    axes[row, 0].set_title("%s  ACF of returns" % asset, fontsize=10)

    plot_pacf(series, lags=MAX_LAG, ax=axes[row, 1], zero=False, method="ywm")
    axes[row, 1].set_title("%s  PACF of returns" % asset, fontsize=10)

    plot_acf(squared, lags=MAX_LAG, ax=axes[row, 2], zero=False)
    axes[row, 2].set_title("%s  ACF of squared returns" % asset, fontsize=10)

    plot_pacf(squared, lags=MAX_LAG, ax=axes[row, 3], zero=False, method="ywm")
    axes[row, 3].set_title("%s  PACF of squared returns" % asset, fontsize=10)

    for column in range(4):
        axes[row, column].set_xlabel("lag in days", fontsize=8)
        axes[row, column].tick_params(labelsize=8)

figure.tight_layout()
plt.show()

## Ljung-Box test

The ACF plots are read one lag at a time, which invites the mistake of noticing
whichever single bar happens to poke out of the band. Ljung-Box tests all lags up
to a chosen horizon jointly, under a null of no autocorrelation at any of them,
so it is the correct instrument for the question "is there any linear structure
here at all".

Three horizons are used. Five days is one week, ten days is two, and 21 days is
roughly one month. The test is run on returns and on squared returns so that the
two can be compared directly on the same scale.

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

LAGS = [5, 10, 21]

rows = []
for asset in assets:
    series = log_return[asset].dropna()
    for label, values in (("returns", series), ("squared returns", series.pow(2))):
        table = acorr_ljungbox(values, lags=LAGS)
        for lag in LAGS:
            rows.append({
                "asset": asset,
                "series": label,
                "lag": lag,
                "lb_stat": float(table.loc[lag, "lb_stat"]),
                "lb_pvalue": float(table.loc[lag, "lb_pvalue"]),
                "rejects_no_autocorrelation": bool(table.loc[lag, "lb_pvalue"] < 0.05),
            })

ljung = pd.DataFrame(rows)
ljung.pivot_table(
    index=["asset", "lag"], columns="series", values=["lb_stat", "lb_pvalue"]
).round(4)

In [ ]:
summary = (
    ljung.groupby(["series", "lag"])["rejects_no_autocorrelation"]
    .agg(assets_rejecting="sum", assets_tested="size")
    .reset_index()
)
summary

### Effect size next to statistical significance

A Ljung-Box p-value answers "is there any linear dependence at all", and with
more than three thousand daily observations that question is answered yes at
levels of dependence far too small to trade. The table above therefore needs a
companion that reports how large the correlations actually are, which is what the
cell below does. It gives the largest absolute autocorrelation over lags 1 to 21
for returns, absolute returns and squared returns, alongside the approximate
significance threshold of 1.96 divided by the square root of the sample size.

In [ ]:
from statsmodels.tsa.stattools import acf

rows = []
for asset in assets:
    series = log_return[asset].dropna()
    rows.append({
        "asset": asset,
        "n": len(series),
        "band": 1.96 / np.sqrt(len(series)),
        "max_abs_acf_returns": float(np.abs(acf(series, nlags=21, fft=True)[1:]).max()),
        "max_abs_acf_absolute": float(np.abs(acf(series.abs(), nlags=21, fft=True)[1:]).max()),
        "max_abs_acf_squared": float(np.abs(acf(series.pow(2), nlags=21, fft=True)[1:]).max()),
    })

magnitudes = pd.DataFrame(rows).set_index("asset")
magnitudes["absolute_over_returns"] = (
    magnitudes["max_abs_acf_absolute"] / magnitudes["max_abs_acf_returns"]
)
magnitudes.round(4)

### Interpretation

Read the two blocks of the Ljung-Box table against each other rather than in
isolation, and read both against the magnitudes above.

The return series rejects the null of no autocorrelation at most assets and
horizons. That is a real result and it should not be talked away, but the
magnitude table shows what is behind it: the largest single autocorrelation over
the first 21 lags is a few hundredths, only marginally above the significance
threshold. Detectable is not the same as exploitable. A signal of that size is
smaller than a realistic round trip cost, and it is the reason the ARIMA
component of this project is expected to select an order close to (0, 0, 0) and
to forecast approximately the training mean, and the reason the directional
baselines are hard to beat.

The squared series rejects the same null far more emphatically, with test
statistics several times larger at every horizon and underlying correlations
several times larger as well. Volatility has structure that is both statistically
clear and large enough to model. This is the standard signature that motivates
conditional variance models, and it is why the volatility target and the GARCH
family are in this project at all.

Two cautions. A rejection at lag 21 says structure exists somewhere in the first
21 lags, not that it is usable after transaction costs. And the test assumes the
series is stationary, which the previous section supports for returns, so
applying it to log prices would produce a number without a meaning.

## Walk forward fold structure

Every result in this project is produced on the folds defined by
`src.splits.walk_forward.PurgedWalkForward` and configured in the `splits` block
of `config/base.yaml`. The folds are computed here on the real feature matrix
index, so the spans drawn below are the spans the models were actually given.

Three properties are worth checking visually.

- The training window expands rather than slides, under the default
  `scheme: expanding`. Each fold trains on everything from `train_start` up to
  its own boundary.
- Test windows do not overlap and move forward one quarter at a time, so no test
  row is ever scored twice.
- There is a gap before each test window. Its size is `horizon + embargo_days`,
  which is six calendar days under the base config. Purging removes training rows
  whose label reaches into the test period, and the embargo removes a further
  buffer because volatility is serially correlated. The same gap is applied at
  the inner boundary between training and validation.

In [ ]:
auxiliary = load_auxiliary(ROOT / "data" / "interim" / "auxiliary.parquet")
matrix = build_features(panel, config, auxiliary)
print("feature matrix:", matrix.X.shape)
matrix.summary()

In [ ]:
splitter = splitter_from_config(config)
folds = splitter.describe(matrix.X.index)

print("scheme:", splitter.scheme)
print("horizon:", splitter.horizon, "days")
print("embargo:", splitter.embargo_days, "days")
print("total gap before each test window:", splitter.gap_days, "days")
print("folds:", len(folds))
folds

In [ ]:
def as_number(value):
    return mdates.date2num(pd.Timestamp(value))


TRAIN_COLOUR = "#3b6ea5"
VAL_COLOUR = "#e08214"
TEST_COLOUR = "#6a3d9a"

figure, axis = plt.subplots(figsize=(13, 0.42 * len(folds) + 2.5))
for _, row in folds.iterrows():
    y = int(row["fold"])
    for start, end, colour in (
        (row["train_start"], row["train_end"], TRAIN_COLOUR),
        (row["val_start"], row["val_end"], VAL_COLOUR),
        (row["test_start"], row["test_end"], TEST_COLOUR),
    ):
        left = as_number(start)
        axis.barh(y, as_number(end) - left, left=left, height=0.62, color=colour)

axis.set_yticks(folds["fold"].tolist())
axis.set_ylabel("fold")
axis.set_xlabel("date")
axis.invert_yaxis()
axis.xaxis.set_major_locator(mdates.YearLocator())
axis.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axis.set_title("Walk forward folds: expanding train, inner validation, held out test")

handles = [
    plt.Rectangle((0, 0), 1, 1, color=TRAIN_COLOUR),
    plt.Rectangle((0, 0), 1, 1, color=VAL_COLOUR),
    plt.Rectangle((0, 0), 1, 1, color=TEST_COLOUR),
]
axis.legend(handles, ["train", "validation", "test"], loc="upper right", fontsize=9)
figure.tight_layout()
plt.show()

### The purge gap at readable scale

The gap is six calendar days against a training span measured in years, so on the
figure above it is under one percent of the axis and cannot be seen. The figure
below zooms on the boundaries of a single fold, where the two gaps are visible as
white space: one between the end of the inner training window and the start of
validation, and one between the end of validation and the start of the test
window.

The annotated distance is seven days rather than the six the config specifies.
That is not an inconsistency. The splitter keeps rows strictly before the
boundary, so the last retained training date falls one day beyond the six day
cut, and the observed spacing between the two blocks is one day wider than the
configured gap.

The counts printed underneath come from the splitter itself. `n_purged` is the
number of training rows dropped because their label overlaps the test window, and
`n_embargoed` is the number dropped by the additional serial correlation buffer.
They are reported separately because they exist for different reasons.

In [ ]:
ZOOM_FOLD = 0
row = folds.loc[folds["fold"] == ZOOM_FOLD].iloc[0]

figure, axes = plt.subplots(1, 2, figsize=(13, 2.8))

boundaries = [
    ("inner train to validation", row["train_end"], row["val_start"], TRAIN_COLOUR, VAL_COLOUR,
     "train", "validation"),
    ("validation to test", row["val_end"], row["test_start"], VAL_COLOUR, TEST_COLOUR,
     "validation", "test"),
]

for axis, (title, left_end, right_start, left_colour, right_colour, left_label, right_label) in zip(
    axes, boundaries
):
    left_end = pd.Timestamp(left_end)
    right_start = pd.Timestamp(right_start)
    window_start = left_end - pd.Timedelta(days=12)
    window_end = right_start + pd.Timedelta(days=12)

    axis.barh(0, as_number(left_end) - as_number(window_start),
              left=as_number(window_start), height=0.5, color=left_colour, label=left_label)
    axis.barh(0, as_number(window_end) - as_number(right_start),
              left=as_number(right_start), height=0.5, color=right_colour, label=right_label)

    gap_days = (right_start - left_end).days
    axis.annotate(
        "gap of %d days" % gap_days,
        xy=(as_number(left_end + (right_start - left_end) / 2), 0.42),
        ha="center", fontsize=9,
    )
    axis.set_xlim(as_number(window_start), as_number(window_end))
    axis.set_ylim(-0.6, 0.75)
    axis.set_yticks([])
    axis.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    axis.tick_params(axis="x", rotation=45, labelsize=8)
    axis.set_title("fold %d, %s" % (ZOOM_FOLD, title), fontsize=10)
    axis.legend(fontsize=8, loc="lower center", ncol=2)

figure.tight_layout()
plt.show()

print("fold %d rows removed by purging:  %d" % (ZOOM_FOLD, row["n_purged"]))
print("fold %d rows removed by embargo:  %d" % (ZOOM_FOLD, row["n_embargoed"]))
print("configured gap: horizon %d + embargo %d = %d days"
      % (splitter.horizon, splitter.embargo_days, splitter.gap_days))

In [ ]:
figure, axis = plt.subplots(figsize=(12, 4))
axis.plot(folds["fold"], folds["n_train"], "o-", label="train rows")
axis.plot(folds["fold"], folds["n_val"], "s-", label="validation rows")
axis.plot(folds["fold"], folds["n_test"], "^-", label="test rows")
axis.set_xlabel("fold")
axis.set_ylabel("rows")
axis.set_title("Sample sizes per fold, the training set grows and the test set does not")
axis.legend(fontsize=9)
figure.tight_layout()
plt.show()

folds[["n_train", "n_val", "n_test", "n_purged", "n_embargoed"]].describe().round(1)

## What this notebook establishes

- Log returns are stationary on every asset, with ADF and KPSS agreeing. Log
  prices are non stationary on most of them, again with both tests agreeing, and
  the verdict column reports any asset where the sample does not deliver that
  result. Modelling returns is therefore not a stylistic preference. It is the
  choice that keeps a single convention across the panel and avoids the inflated
  fit that comes from regressing a trending level on its own past.
- Returns do show statistically detectable linear autocorrelation at one week,
  two week and one month horizons, but the correlations behind those rejections
  are a few hundredths. A directional model has to find non linear or cross
  sectional structure, because the linear structure that exists is too small to
  use.
- Squared returns show autocorrelation that is both significant and several times
  larger at every horizon tested. The volatility target is the part of this
  problem with a usable signal in it.
- Results are reported on non overlapping quarterly test windows with an
  expanding training window and a six day gap that removes both label overlap and
  the serial correlation buffer around each boundary.